In [1]:
import time

import pandas as pd

from src.data.load_raw import load_raw_data
from src.data.preprocess import (
    extract_xy,
    merge_features_and_labels,
    standardize_splits,
)
from src.data.split import split_labeled_transactions
from src.evaluation.metrics import calculate_binary_metrics
from src.evaluation.threshold import find_best_f1_threshold
from sklearn.model_selection import TimeSeriesSplit
from src.models.classic import (
    build_dummy_classifier,
    build_logistic_regression,
    build_random_forest,
    build_xgboost,
)

import optuna
from sklearn.model_selection import cross_val_score
import numpy as np

import warnings
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=FutureWarning, module="sklearn.linear_model")
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn.linear_model")

warnings.filterwarnings("ignore", category=ConvergenceWarning)

/home/piotrs/bitcoin-transaction-classification/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
features_df, classes_df, edges_df = load_raw_data()

transactions_df = merge_features_and_labels(
    features_df,
    classes_df,
)

splits = split_labeled_transactions(transactions_df)

x_train, y_train = extract_xy(
    splits.train,
    feature_set="all",
)

x_validation, y_validation = extract_xy(
    splits.validation,
    feature_set="all",
)

x_test, y_test = extract_xy(
    splits.test,
    feature_set="all",
)

(
    scaler,
    x_train_scaled,
    x_validation_scaled,
    x_test_scaled,
) = standardize_splits(
    x_train,
    x_validation,
    x_test,
)

In [ ]:
def objective1(trial, X, y):
    solver = trial.suggest_categorical("solver", ["lbfgs", "liblinear", "saga"])

    if solver == "lbfgs":
        reg_type = trial.suggest_categorical("reg_lbfgs", ["l2", "none"])
        if reg_type == "none":
            C = np.inf
            l1_ratio = None
        else:
            C = trial.suggest_float("C_lbfgs", 1e-4, 1e4, log=True)
            l1_ratio = 0.0

    elif solver == "liblinear":
        reg_type = trial.suggest_categorical("reg_lib", ["l1", "l2"])
        C = trial.suggest_float("C_lib", 1e-4, 1e4, log=True)
        l1_ratio = 1.0 if reg_type == "l1" else 0.0

    else:
        reg_type = trial.suggest_categorical("reg_saga", ["l1", "l2", "elasticnet", "none"])
        if reg_type == "none":
            C = np.inf
            l1_ratio = None
        else:
            C = trial.suggest_float("C_saga", 1e-4, 1e4, log=True)
            if reg_type == "l1":
                l1_ratio = 1.0
            elif reg_type == "l2":
                l1_ratio = 0.0
            else:
                l1_ratio = trial.suggest_float("l1_ratio_saga", 0.01, 0.99)

    max_iter = trial.suggest_int("max_iter", 500, 2000, step=100)

    model = build_logistic_regression(
        solver=solver,
        C=C,
        l1_ratio=l1_ratio,
        max_iter=max_iter
    )

    scores = cross_val_score(
        model,
        X,
        y,
        cv=5,
        scoring="f1_macro",
        n_jobs=-1
    )

    return np.mean(scores)


study = optuna.create_study(direction="maximize", study_name="Logistic_Regression_Tuning")

study.optimize(lambda trial: objective1(trial, x_train, y_train), n_trials=50, show_progress_bar=True)

print("\n--- Optimization Finished ---")
print(f"Best Trial Score: {study.best_value:.4f}")
print("Best Parameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

In [ ]:
def objective2(trial, X, y):
    n_estimators = trial.suggest_int("n_estimators", 100, 1500, step=100)

    depth_option = trial.suggest_categorical("depth_type", ["unlimited", "constrained"])
    if depth_option == "unlimited":
        max_depth = None
    else:
        max_depth = trial.suggest_int("max_depth", 5, 100) # Much deeper upper limit

    min_samples_split = trial.suggest_int("min_samples_split", 2, 100)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 50)

    max_features = trial.suggest_categorical("max_features", ["sqrt", "log2", None])
    criterion = trial.suggest_categorical("criterion", ["gini", "entropy", "log_loss"])

    model = build_random_forest(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        criterion=criterion
    )

    scores = cross_val_score(
        model,
        X,
        y,
        cv=5,
        scoring="f1_macro",
        n_jobs=-1
    )

    return np.mean(scores)

# --- Execution ---
study = optuna.create_study(direction="maximize", study_name="Random_Forest_Ultimate_Tuning")

study.optimize(
    lambda trial: objective2(trial, x_train, y_train),
    n_trials=200,
    show_progress_bar=True
)

print("\n--- Optimization Finished ---")
print(f"Best Trial Score: {study.best_value:.4f}")
print("Best Parameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

In [ ]:
def objective3(trial, X, y):
    learning_rate = trial.suggest_float("learning_rate", 1e-3, 0.3, log=True)
    n_estimators = trial.suggest_int("n_estimators", 100, 1000, step=100)

    max_depth = trial.suggest_int("max_depth", 3, 12)
    min_child_weight = trial.suggest_int("min_child_weight", 1, 10)

    subsample = trial.suggest_float("subsample", 0.5, 1.0)
    colsample_bytree = trial.suggest_float("colsample_bytree", 0.5, 1.0)

    gamma = trial.suggest_float("gamma", 0.0, 5.0)
    reg_alpha = trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True)
    reg_lambda = trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True)

    scale_pos_weight = trial.suggest_float("scale_pos_weight", 5.0, 15.0)

    model = build_xgboost(
        learning_rate=learning_rate,
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_child_weight=min_child_weight,
        gamma=gamma,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        reg_alpha=reg_alpha,
        reg_lambda=reg_lambda,
        scale_pos_weight=scale_pos_weight
    )

    tscv = TimeSeriesSplit(n_splits=5)

    scores = cross_val_score(
        model,
        X,
        y,
        cv=tscv,
        scoring="recall",
        n_jobs=-1
    )

    return np.mean(scores)

# --- Execution ---
study = optuna.create_study(direction="maximize", study_name="Random_Forest_Ultimate_Tuning")

study.optimize(
    lambda trial: objective3(trial, x_train, y_train),
    n_trials=200,
    show_progress_bar=True
)

print("\n--- Optimization Finished ---")
print(f"Best Trial Score: {study.best_value:.4f}")
print("Best Parameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

[I 2026-06-07 19:59:22,999] A new study created in memory with name: Random_Forest_Ultimate_Tuning
  0%|          | 0/200 [00:00<?, ?it/s]/usr/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/usr/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/usr/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for 

[I 2026-06-07 19:59:29,767] Trial 0 finished with value: 0.6282824539968294 and parameters: {'learning_rate': 0.0014133700215610602, 'n_estimators': 600, 'max_depth': 4, 'min_child_weight': 4, 'subsample': 0.8720364558096396, 'colsample_bytree': 0.8336733057960206, 'gamma': 2.4536785004487376, 'reg_alpha': 2.6392193643831073e-07, 'reg_lambda': 4.956805971942923, 'scale_pos_weight': 8.126181072020586}. Best is trial 0 with value: 0.6282824539968294.


/usr/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/usr/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/usr/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/usr/lib/pyth

[I 2026-06-07 19:59:37,196] Trial 1 finished with value: 0.7850864719274895 and parameters: {'learning_rate': 0.04364134631618033, 'n_estimators': 1000, 'max_depth': 3, 'min_child_weight': 6, 'subsample': 0.9786267083444193, 'colsample_bytree': 0.9218660335064892, 'gamma': 1.378027236461734, 'reg_alpha': 0.11340741234851816, 'reg_lambda': 1.892843147374603e-07, 'scale_pos_weight': 14.573829832363906}. Best is trial 1 with value: 0.7850864719274895.


/usr/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/usr/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/usr/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/usr/lib/pyth